# DTDP - Dynamic Train Dispatching Problem

**A minimal NP-hard problem for Japan railway scheduling**

This notebook walks through:
1. Problem structure (Graph G = (V, E))
2. Generating instances
3. Evaluating solutions
4. Visualizing results
5. ShinkaEvolve integration

---
## 1. The Problem

```
Given:  Railway network G = (V, E) + trains
Find:   Departure time for each train
Goal:   Minimize conflicts + delays
```

**Why NP-hard?** Single-track segments mean trains conflict. Deciding who waits where creates exponential possibilities.

In [ ]:
# Import modules
from problem import generate, Instance, RailGraph, Train
from fitness import evaluate, Schedule
from solver import naive, greedy_fcfs, greedy_priority, direction_wave, benchmark, print_benchmark
from visualize import show_graph, show_trains, show_schedule, show_timespace, report
from shinka import DTDPProblem

---
## 2. Graph Structure: G = (V, E)

The railway network is a weighted graph:
- **V** = stations (nodes)
- **E** = track segments with travel times (edges)

In [ ]:
# Generate a simple instance
inst = generate(num_stations=5, num_trains=6, seed=42)

print(show_graph(inst.graph))

In [ ]:
# Graph internals
print(f"|V| = {len(inst.graph.vertices)} stations")
print(f"|E| = {len(inst.graph.edges)} directed edges")
print()
print("Edges (bidirectional):")
seen = set()
for (u, v), t in inst.graph.edges.items():
    edge = tuple(sorted([u, v]))
    if edge not in seen:
        seen.add(edge)
        print(f"  {u} ↔ {v} : {t} time units")

In [ ]:
# Graphviz DOT format (for rendering)
print(inst.graph.to_dot())

---
## 3. Trains

Each train has:
- `origin` → `destination` (path computed via BFS)
- `earliest` departure time
- `priority` (higher = more important)

In [ ]:
print(show_trains(inst))

In [ ]:
# Train details
for t in inst.trains:
    path = inst.graph.path(t.origin, t.destination)
    print(f"Train {t.id}:")
    print(f"  Route: {' → '.join(map(str, path))}")
    print(f"  Earliest departure: {t.earliest}")
    print(f"  Priority: {t.priority}")
    print()

---
## 4. Solution Format

A solution is simply: **when does each train depart?**

```python
schedule = {
    0: 5,   # Train 0 departs at t=5
    1: 8,   # Train 1 departs at t=8
    ...
}
```

In [ ]:
# Naive solution: everyone departs at earliest time
naive_schedule = naive(inst)
print("Naive schedule:")
print(naive_schedule)

---
## 5. Fitness Evaluation

```
fitness = delay + conflicts + makespan
          (lower = better)
```

| Component | Weight | Description |
|-----------|--------|-------------|
| Delay | 1.0 | `(departure - earliest) × priority` |
| Conflicts | 50.0 | Trains on same segment at same time |
| Makespan | 0.1 | Total schedule duration |

In [ ]:
# Evaluate naive solution
result = evaluate(inst, naive_schedule)

print(f"Total Score: {result.score} (lower = better)")
print(f"  Delay:     {result.delay}")
print(f"  Conflicts: {result.conflicts} ({result.num_conflicts} conflicts)")
print(f"  Makespan:  {result.makespan}")

---
## 6. Visualization

In [ ]:
# Schedule timeline
print(show_schedule(inst, naive_schedule))

In [ ]:
# Time-space diagram
# Rows = stations, Columns = time
# X = conflict!
print(show_timespace(inst, naive_schedule))

---
## 7. Compare Baseline Solvers

In [ ]:
results = benchmark(inst)
print_benchmark(results)

In [ ]:
# Best solution visualization
best_name = min(results, key=lambda k: results[k].score)
best_solver = {"naive": naive, "greedy_fcfs": greedy_fcfs, 
               "greedy_priority": greedy_priority, "direction_wave": direction_wave}[best_name]

print(f"Best: {best_name}\n")
print(show_timespace(inst, best_solver(inst)))

---
## 8. Harder Instance

In [ ]:
# More trains = more conflicts
hard_inst = generate(num_stations=7, num_trains=12, seed=123)

print(show_graph(hard_inst.graph))
print()
print(show_trains(hard_inst))

In [ ]:
print("Baseline comparison on harder instance:\n")
hard_results = benchmark(hard_inst)
print_benchmark(hard_results)

---
## 9. Branching Network

In [ ]:
# Non-linear topology
branch_inst = generate(num_stations=8, num_trains=10, topology="branching", seed=42)

print(show_graph(branch_inst.graph))
print()
print(show_trains(branch_inst))

In [ ]:
print(branch_inst.graph.to_dot())

---
## 10. ShinkaEvolve Integration

The `DTDPProblem` class provides the interface for algorithm evolution.

In [ ]:
# Create problem with difficulty level
problem = DTDPProblem(difficulty="medium")

print(f"Test instances: {len(problem.instances)}")
for i, inst in enumerate(problem.instances):
    print(f"  [{i}] {inst.num_stations} stations, {inst.num_trains} trains")

In [ ]:
# Get solver template
print(problem.get_template())

In [ ]:
# Evaluate baseline
score = problem.evaluate(direction_wave)
print(f"Direction wave score: {score}")

In [ ]:
# Get feedback (for LLM learning)
print(problem.get_feedback())

---
## 11. Write Your Own Solver

Try to beat the baselines!

In [ ]:
def my_solver(inst: Instance) -> Schedule:
    """
    Your algorithm here!
    
    Available:
        inst.graph.path(origin, dest) -> [stations] or None
        inst.graph.travel_time(u, v) -> int
        inst.graph.neighbors(v) -> [stations]
        inst.trains -> List[Train]
        inst.horizon -> int
    
    Train attributes:
        train.id, train.origin, train.destination
        train.earliest, train.priority
    """
    schedule = {}
    
    # Example: sort by priority, add spacing
    sorted_trains = sorted(inst.trains, key=lambda t: (-t.priority, t.earliest))
    
    time_cursor = 0
    for train in sorted_trains:
        schedule[train.id] = max(train.earliest, time_cursor)
        time_cursor = schedule[train.id] + 3
    
    return schedule

# Test it
score = problem.evaluate(my_solver)
print(f"My solver score: {score}")
print()
print(problem.get_feedback())

---
## 12. Full Benchmark

In [ ]:
# Test all solvers on full test suite
full_problem = DTDPProblem(difficulty="full")

print("Full benchmark (8 instances):\n")
for name, solver in [("naive", naive), ("greedy_fcfs", greedy_fcfs), 
                      ("greedy_priority", greedy_priority), 
                      ("direction_wave", direction_wave),
                      ("my_solver", my_solver)]:
    score = full_problem.evaluate(solver)
    print(f"{name:<20} {score:>10.1f}")

---
## Summary

**Problem:** Schedule trains to minimize conflicts on single-track railways

**Graph:** G = (V, E) where V = stations, E = track segments

**Solution:** `Dict[train_id, departure_time]`

**Fitness:** `delay + conflicts×50 + makespan×0.1` (lower = better)

**ShinkaEvolve:**
```python
problem = DTDPProblem(difficulty="medium")
score = problem.evaluate(my_solver)
feedback = problem.get_feedback()
```

**Goal:** Beat `direction_wave` baseline!